# 02 — Evaluation: quality-aware late fusion of visual and thermal detectors

**Visual–thermal monkey detection across times of day** · [Repository](https://github.com/ellail-mardi/monkey-detection)

This notebook evaluates two YOLO11n detectors (one per modality) and their late fusion with Weighted Box Fusion (WBF), using either fixed modality weights or the adaptive weights of **Adaptive Late Fusion (ALF)**:

* **VQE** (visual quality estimator): mean HSV *V* of the RGB frame, $\mu_V$ → visual quality *f*($\mu_V$).
* **TQE** (thermal quality estimator): global Michelson contrast of the grey thermal frame, $C_T$ → thermal quality *g*($C_T$).
* Weights $w_V = f/(f+g)$, $w_T = g/(f+g)$ rescale detector confidences before WBF.

**Inputs.** The cloned repository (test pairs, training pairs, optional source-only weights) and `cv/oof_preds.pkl` produced by `01_session_grouped_cv_training.ipynb` (3-fold, session-grouped, out-of-fold predictions; may also be shipped in the repository).

**Outputs** (`results/`): overall performance with session-level bootstrap intervals; paired comparisons; results by time window; ablation (VQE only / TQE only); quality-estimator statistics; fixed-weight sweeps with oracle headroom; per-window versus pooled evaluation; source-only versus cross-validated detectors; sensitivity analysis; latency; qualitative figure; per-image tables.

**Evaluation conventions.** mAP@0.5 and mAP@0.5:0.95 use greedy confidence-ordered matching and 101-point interpolated AP (COCO style). Precision/recall are reported at the F1-optimal confidence threshold (IoU 0.5) of each method. Results are **pooled** (all pairs of a window, or of the whole test set, are ranked together, as with a single global threshold). Confidence intervals use a stratified **session-level** bootstrap. Oracle weights and weight sweeps use test labels and are descriptive bounds only; the max-normalised variant is an exploratory, post hoc diagnostic.

**Run:** Runtime → Change runtime type → GPU (recommended), then Run all.

## 1. Install

In [ ]:
!pip -q install -U ultralytics ensemble-boxes openpyxl

## 2. Configuration and data discovery

In [ ]:
import os, re, io, sys, json, time, shutil, hashlib, pickle, random, warnings, platform, zipfile, subprocess, itertools
from pathlib import Path
from collections import OrderedDict, Counter, defaultdict
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ----------------------------- Locations -----------------------------
REPO_URL = "https://github.com/ellail-mardi/monkey-detection"
REPO_DIR = Path(os.environ.get("ALF_REPO", "/content/monkey-detection"))   # where the repository is (or will be cloned)
USE_GOOGLE_DRIVE = False        # True: write generated files (cv/, results/) to Google Drive so they survive a Colab disconnect
DRIVE_OUT = "/content/drive/MyDrive/monkey-detection-outputs"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive"); OUTPUT_ROOT = Path(DRIVE_OUT)
else:
    OUTPUT_ROOT = REPO_DIR
CV_DIR, RESULTS_DIR = OUTPUT_ROOT / "cv", OUTPUT_ROOT / "results"
CV_DIR.mkdir(parents=True, exist_ok=True); RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------- Data layout (auto-detected) -----------------------------
# Expected layout (folder names are matched case-insensitively; Malay names are accepted as aliases):
#   <data root>/test/{dawn|subuh, midday|tengahari, dusk|senja, night|malam}/{visual, thermal, labels}
#   <training folder>/{visual, thermal, labels}      e.g. train/
#   weights/visual_best.pt, weights/thermal_best.pt  (source-only detectors, optional)
# Override any of the following with an explicit Path if auto-detection fails.
DATA_ROOT = None; TRAIN_DIR = None; TEST_DIR = None; W_VIS = None; W_THR = None

COND_ALIASES = OrderedDict([("dawn", ["dawn", "subuh"]), ("midday", ["midday", "tengahari"]),
                            ("dusk", ["dusk", "senja"]), ("night", ["night", "malam"])])
COND_MAP = OrderedDict([("dawn", "Dawn"), ("midday", "Midday"), ("dusk", "Dusk"), ("night", "Night")])
def _child(d, names):
    if d is None or not Path(d).exists(): return None
    low = {p.name.lower(): p for p in Path(d).iterdir() if p.is_dir()}
    for n in names:
        if n.lower() in low: return low[n.lower()]
    return None
def _walk_dirs(root, max_depth):
    queue = [(Path(root), 0)]
    while queue:
        d, k = queue.pop(0); yield d
        if k < max_depth:
            queue += [(p, k + 1) for p in sorted(d.iterdir()) if p.is_dir() and not p.name.startswith(".")]
def locate():
    global DATA_ROOT, TRAIN_DIR, TEST_DIR, W_VIS, W_THR
    if TEST_DIR is None:
        if DATA_ROOT is None:
            for d in _walk_dirs(REPO_DIR, 3):
                t = _child(d, ["test"])
                if t is not None and all(_child(t, a) is not None for a in COND_ALIASES.values()): DATA_ROOT = d; break
        assert DATA_ROOT is not None, "Could not find a 'test' folder with the four time-of-day sub-folders. Set DATA_ROOT / TEST_DIR manually."
        TEST_DIR = _child(DATA_ROOT, ["test"])
    if TRAIN_DIR is None:
        base = DATA_ROOT if DATA_ROOT is not None else TEST_DIR.parent
        for d in _walk_dirs(base, 2):
            if d == TEST_DIR or TEST_DIR in d.parents: continue
            if _child(d, ["visual"]) is not None and _child(d, ["thermal"]) is not None: TRAIN_DIR = d; break
    if W_VIS is None: W_VIS = next(iter(sorted(REPO_DIR.rglob("visual_best.pt"))), None)
    if W_THR is None: W_THR = next(iter(sorted(REPO_DIR.rglob("thermal_best.pt"))), None)
locate()
print("Repository      :", REPO_DIR); print("Test folder     :", TEST_DIR); print("Training folder :", TRAIN_DIR)
print("Source weights  :", W_VIS, "|", W_THR); print("Outputs         :", CV_DIR, "|", RESULTS_DIR)

# ----------------------------- ALF parameters (fixed a priori, not tuned on test data) -----------------------------
ALFP = dict(tau_dark=0.2, tau_bright=0.85, QV_low=0.3, QV_high=0.8,      # VQE
            tau_cross=0.25, QT_low=0.3, QT_high=0.8)                      # TQE
TQE_PERCENTILE = None    # None = true I_max / I_min; e.g. (1, 99) = percentile-based, robust to extreme pixels (not used in the reported analysis)

# ----------------------------- Detection and fusion settings -----------------------------
IMGSZ, CONF_PRED, NMS_IOU, MAX_DET, BATCH = 575, 0.001, 0.6, 100, 16     # low confidence threshold, as is standard for mAP
WBF_IOU, WBF_SKIP, WBF_CONF_TYPE = 0.55, 0.0, "avg"
MAX_DET_EVAL = 100
GROUPS = OrderedDict([("Dawn / Dusk", ["dawn", "dusk"]), ("Midday", ["midday"]), ("Night", ["night"])])   # pooled time windows

# ----------------------------- Statistics and runtime options -----------------------------
SEED = 42
N_BOOT = 1000                                  # bootstrap resamples
BOOT_UNIT = "session"                          # "session" (recommended: frames of a session are correlated) or "image"
WGRID = np.round(np.linspace(0, 1, 21), 2)     # grid of visual weights w_V for the static-weight search and sweeps
RUN_SENSITIVITY = True                         # ~5-10 min
RUN_LATENCY = True
QUAL_MODE = "max_gain"                         # qualitative frames: "max_gain" (largest ALF-vs-fixed difference) or "random"
FORCE_RERUN = False                            # True: ignore cached source-only predictions
DECIMALS = 1

random.seed(SEED); np.random.seed(SEED)
SINGLE_CLASS = True; NC = 1
OOF_FILE = next((p for p in (CV_DIR / "oof_preds.pkl", REPO_DIR / "cv" / "oof_preds.pkl") if p.exists()), None)
assert OOF_FILE is not None, ("cv/oof_preds.pkl not found. Run 01_session_grouped_cv_training.ipynb first (or place the file in <repo>/cv/).")
print("Out-of-fold predictions:", OOF_FILE)

## 3. Data checks: pairs, annotations, leakage

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
DUP_KEYS = []
def _pair_key(p):
    # pair key = session_frame, e.g. v19_00001.png / t19_00001.png -> "19_00001" (robust to Roboflow-style suffixes)
    m = re.search(r"(\d+_\d+)", p.stem)
    return m.group(1) if m else re.sub(r"^[A-Za-z]+[_\-]?", "", p.stem)
def _list_images(d):
    out = {}
    if d is None or not Path(d).exists(): return out
    for p in sorted(Path(d).iterdir()):
        if p.suffix.lower() not in IMG_EXT: continue
        k = _pair_key(p)
        if k in out: DUP_KEYS.append(f"{d}: {out[k].name} vs {p.name}")
        out[k] = p
    return out
def session_of(p):
    m = re.match(r"[A-Za-z]*(\d+)_", p.stem); return m.group(1) if m else "?"
def load_boxes(p):
    """YOLO label file -> normalised xyxy boxes (polygons are converted to their bounding box)."""
    if p is None or not Path(p).exists(): return np.zeros((0, 4), np.float32)
    B = []
    for ln in Path(p).read_text().strip().splitlines():
        t = ln.split()
        if len(t) == 5:
            x, y, w, h = map(float, t[1:]); B.append([x - w/2, y - h/2, x + w/2, y + h/2])
        elif len(t) > 5:
            xs, ys = np.array(t[1::2], float), np.array(t[2::2], float); B.append([xs.min(), ys.min(), xs.max(), ys.max()])
    return np.clip(np.array(B, np.float32).reshape(-1, 4), 0, 1)
def _iou1(a, b):
    iw, ih = min(a[2], b[2]) - max(a[0], b[0]), min(a[3], b[3]) - max(a[1], b[1]); inter = max(iw, 0) * max(ih, 0)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)
GT_DEDUP_IOU = 0.95        # after merging all classes into one, drop duplicate boxes (e.g. a visual and a thermal box on the same animal)
def dedup(b):
    keep = []
    for k in range(len(b)):
        if all(_iou1(b[k], b[j]) < GT_DEDUP_IOU for j in keep): keep.append(k)
    return b[keep]
def collect_pairs(d, cond, split):
    """Pair visual/thermal frames of a folder by session_frame key; attach (single-class) ground-truth boxes."""
    d = Path(d); vis, thr = _list_images(_child(d, ["visual"])), _list_images(_child(d, ["thermal"])); lab = _child(d, ["labels"])
    recs = []
    for k in sorted(set(vis) & set(thr)):
        lv = lab / f"{vis[k].stem}.txt" if lab else None; lt = lab / f"{thr[k].stem}.txt" if lab else None
        lv_ok, lt_ok = bool(lv and lv.exists()), bool(lt and lt.exists())
        label_v = lv if lv_ok else (lt if lt_ok else None)
        recs.append(dict(split=split, cond=cond, key=k, vis=vis[k], thr=thr[k], session=session_of(vis[k]),
                         boxes_V=dedup(load_boxes(label_v)),                       # boxes for the visual frame
                         boxes_T=dedup(load_boxes(lt if lt_ok else label_v)),      # boxes for the thermal frame (own file if present)
                         has_thermal_label_file=lt_ok and lt != label_v))
    return recs, sorted(set(vis) - set(thr)), sorted(set(thr) - set(vis))
def collect_test():
    recs, problems = [], []
    for c, aliases in COND_ALIASES.items():
        r, ov, ot = collect_pairs(_child(TEST_DIR, aliases), c, "test"); recs += r
        if ov or ot: problems.append(f"[test/{c}] unpaired frames -> visual only: {ov} | thermal only: {ot}")
    return recs, problems

ALL, problems = collect_test()
for i, r in enumerate(ALL):
    r["i"] = i; r["gt_boxes"] = r["boxes_V"]; r["gt_cls"] = np.zeros(len(r["boxes_V"]), int)     # one set of boxes for both frames; single class
TEST_IDX = list(range(len(ALL)))
POS = {c: [k for k, r in enumerate(ALL) if r["cond"] == c] for c in COND_MAP}       # positions of each window within the test list
SESS = [r["session"] for r in ALL]

POOL_RECS = []
if TRAIN_DIR is not None: POOL_RECS, ov, ot = collect_pairs(TRAIN_DIR, "train", "train")

rows = []
for c, cn in COND_MAP.items():
    rs = [r for r in ALL if r["cond"] == c]
    v0, t0 = cv2.imread(str(rs[0]["vis"])), cv2.imread(str(rs[0]["thr"]), cv2.IMREAD_UNCHANGED)
    rows.append(dict(Subset="Test", Window=cn, Pairs=len(rs), Sessions=len({r["session"] for r in rs}), Instances=int(sum(len(r["gt_cls"]) for r in rs)),
                     Instances_per_pair=round(float(np.mean([len(r["gt_cls"]) for r in rs])), 2), Pairs_without_boxes=int(sum(len(r["gt_cls"]) == 0 for r in rs)),
                     Visual_size=f"{v0.shape[1]}x{v0.shape[0]}", Thermal_size=f"{t0.shape[1]}x{t0.shape[0]}"))
if POOL_RECS:
    v0, t0 = cv2.imread(str(POOL_RECS[0]["vis"])), cv2.imread(str(POOL_RECS[0]["thr"]), cv2.IMREAD_UNCHANGED)
    rows.append(dict(Subset="Training", Window="(all)", Pairs=len(POOL_RECS), Sessions=len({r["session"] for r in POOL_RECS}),
                     Instances=f"{sum(len(r['boxes_V']) for r in POOL_RECS)} visual / {sum(len(r['boxes_T']) for r in POOL_RECS)} thermal",
                     Instances_per_pair=round(float(np.mean([len(r["boxes_V"]) for r in POOL_RECS])), 2),
                     Pairs_without_boxes=int(sum(len(r["boxes_V"]) == 0 for r in POOL_RECS)), Visual_size=f"{v0.shape[1]}x{v0.shape[0]}", Thermal_size=f"{t0.shape[1]}x{t0.shape[0]}"))
T_DATA = pd.DataFrame(rows); display(T_DATA)
print(f"Test: {len(ALL)} pairs, {sum(len(r['gt_cls']) for r in ALL)} instances, {len(set(SESS))} sessions")
if DUP_KEYS: problems.append(f"duplicate pair keys ({len(DUP_KEYS)}), e.g. {DUP_KEYS[:3]}")
for p in problems: print("WARNING:", p)

# Leakage checks: identical files between test and training data; duplicate test frames
def md5(p): return hashlib.md5(Path(p).read_bytes()).hexdigest()
h_test = {md5(r[k]) for r in ALL for k in ("vis", "thr")}
h_train = {md5(r[k]) for r in POOL_RECS for k in ("vis", "thr")}
print("Identical files shared by test and training data:", len(h_test & h_train) if POOL_RECS else "n/a (no training folder)")
_hv = defaultdict(list)
for r in ALL: _hv[md5(r["vis"])].append(f'{r["cond"]}/{r["vis"].name}')
print("Duplicate visual test frames:", [v for v in _hv.values() if len(v) > 1] or "none")
if POOL_RECS:
    ts = {r["session"] for r in POOL_RECS} & set(SESS)
    if ts: print(f"NOTE: session IDs {sorted(ts)} occur in both training and test folders (fine if numbering restarts per folder).")
    best = [max(_iou1(a, b) for b in r["boxes_T"]) for r in POOL_RECS if r["has_thermal_label_file"] for a in r["boxes_V"] if len(r["boxes_T"])]
    if best: print(f"Training pairs: median IoU between the visual and the thermal box of the same pair = {np.median(best):.2f} (low => the two cameras are not aligned, hence separate annotations)")

## 4. Source-only detectors (optional) and inference

In [ ]:
import torch
from ultralytics import YOLO
from importlib.metadata import version as _v
_uv = tuple(int(x) for x in re.findall(r"\d+", _v("ultralytics"))[:2])
assert _uv >= (8, 3), f"ultralytics {_v('ultralytics')} is too old for YOLO11 (need >= 8.3)"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("ultralytics", _v("ultralytics"), "| device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU")

def predict_paths(model, paths):
    out = []
    for i in range(0, len(paths), BATCH):
        res = model.predict([str(p) for p in paths[i:i + BATCH]], imgsz=IMGSZ, conf=CONF_PRED, iou=NMS_IOU, max_det=MAX_DET, device=DEVICE, verbose=False)
        for r in res:
            b = r.boxes; cf = b.conf.cpu().numpy().astype(np.float32)
            out.append((b.xyxyn.cpu().numpy().astype(np.float32), cf, np.zeros(len(cf), int)))      # single class
    return out

# Source-only detectors (trained on the original training pairs only) -> used only for the source-vs-CV comparison
HAVE_SOURCE = W_VIS is not None and W_THR is not None
PRED_SRC = None
if HAVE_SOURCE:
    sig = hashlib.md5(json.dumps([str(W_VIS), Path(W_VIS).stat().st_size, str(W_THR), Path(W_THR).stat().st_size, IMGSZ, CONF_PRED, NMS_IOU, MAX_DET,
                                  [r["vis"].name for r in ALL]]).encode()).hexdigest()
    CACHE = RESULTS_DIR / "source_only_preds_cache.pkl"
    if CACHE.exists() and not FORCE_RERUN:
        c = pickle.load(open(CACHE, "rb"))
        if c.get("sig") == sig: PRED_SRC = c["pred"]; print("Source-only predictions loaded from cache")
    if PRED_SRC is None:
        t0 = time.time(); mV, mT = YOLO(str(W_VIS)), YOLO(str(W_THR))
        PRED_SRC = {"V": predict_paths(mV, [r["vis"] for r in ALL]), "T": predict_paths(mT, [r["thr"] for r in ALL])}
        pickle.dump({"sig": sig, "pred": PRED_SRC}, open(CACHE, "wb")); print(f"Source-only inference done in {time.time()-t0:.0f}s")
else:
    print("Source-only weights not found: the source-vs-CV comparison will be skipped.")

# Cross-validated (out-of-fold) predictions: the main analysis
OOF = pickle.load(open(OOF_FILE, "rb")); OOF_BY_NAME = {k.split("/")[-1]: v for k, v in OOF.items()}
PRED = {"V": [], "T": []}
for r in ALL:
    o = OOF_BY_NAME[r["vis"].name]
    for m in ("V", "T"): PRED[m].append((o[m][0], o[m][1], np.zeros_like(o[m][2])))
    r["fold"], r["session"] = int(o["fold"]), o["session"]
SESS = [r["session"] for r in ALL]
print(f"Out-of-fold predictions loaded: {len(ALL)} pairs, {len({r['fold'] for r in ALL})} folds")
print("Mean boxes per frame  visual:", np.mean([len(p[1]) for p in PRED["V"]]).round(1), "| thermal:", np.mean([len(p[1]) for p in PRED["T"]]).round(1))

## 5. Quality estimators: $\mu_V$ and $C_T$ for every pair

In [ ]:
def mu_v(img_bgr):                          # mean of the HSV V channel, scaled to [0, 1]
    return float(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)[..., 2].mean() / 255.0)
def thermal_gray01(p):                      # thermal frames are single-channel grey: read directly, scale by 1/255 (no per-frame min-max stretching)
    return cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
def michelson(g, pct=TQE_PERCENTILE):
    lo, hi = (np.percentile(g, pct[0]), np.percentile(g, pct[1])) if pct else (g.min(), g.max())
    return float((hi - lo) / (hi + lo + 1e-8))
for r in ALL:
    r["muV"] = mu_v(cv2.imread(str(r["vis"]), cv2.IMREAD_COLOR)); r["CT"] = michelson(thermal_gray01(r["thr"]))
QDF = pd.DataFrame([dict(frame=r["vis"].name, window=COND_MAP[r["cond"]], session=r["session"], muV=r["muV"], CT=r["CT"]) for r in ALL])
display(QDF.groupby("window", sort=False)[["muV", "CT"]].agg(["mean", "std", "min", "max"]).round(3))
trig_cross = (QDF.CT < ALFP["tau_cross"]).mean()
print(f"Share of frames with C_T < tau_cross ({ALFP['tau_cross']}): {trig_cross*100:.1f}%")
if trig_cross == 0:
    print("NOTE: the TQE never fires with the current tau_cross, so the 'TQE only' ablation equals fixed-weight fusion.")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8)); labels = list(COND_MAP.values())
for a, col, lines, ttl in [(ax[0], "muV", [("tau_dark", ALFP["tau_dark"]), ("tau_bright", ALFP["tau_bright"])], "Visual brightness $\\mu_V$ (VQE)"),
                           (ax[1], "CT", [("tau_cross", ALFP["tau_cross"])], "Thermal Michelson contrast $C_T$ (TQE)")]:
    data = [QDF[QDF.window == w][col].values for w in labels]
    a.boxplot(data, showfliers=False); a.set_xticklabels(labels)
    for k, d in enumerate(data): a.scatter(np.random.normal(k + 1, 0.05, len(d)), d, s=8, alpha=.5)
    for n, v in lines: a.axhline(v, color="r", ls="--", lw=1); a.text(0.55, v, n, color="r", fontsize=8, va="bottom")
    a.set_title(ttl, fontsize=10); a.set_ylim(0, 1.02); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig_quality_estimator_distributions.png", dpi=300); plt.show()

## 6. Core functions: adaptive weights, weighted box fusion, evaluation

In [ ]:
from ensemble_boxes import weighted_boxes_fusion

def alf_weights(muV, CT, p, mode="full"):
    """Adaptive weights (w_V, w_T). mode: 'full' | 'vqe' (thermal quality fixed high) | 'tqe' (visual quality fixed high) | 'norm' (post hoc, max-normalised)."""
    f = p["QV_low"] if (muV < p["tau_dark"] or muV > p["tau_bright"]) else p["QV_high"]
    g = p["QT_low"] if CT < p["tau_cross"] else p["QT_high"]
    if mode == "vqe": g = p["QT_high"]
    if mode == "tqe": f = p["QV_high"]
    if mode == "norm": m = max(f, g); return f / m, g / m
    s = f + g
    return f / s, g / s

_fuse_cache = {}
def fuse(i, wV, wT):
    """Rescale detector confidences by the modality weights (s' = w * s), then run standard WBF (no extra per-model weights)."""
    key = (i, round(wV, 5), round(wT, 5))
    if key in _fuse_cache: return _fuse_cache[key]
    pV, pT = PRED["V"][i], PRED["T"][i]
    bl = [np.clip(pV[0], 0, 1).tolist(), np.clip(pT[0], 0, 1).tolist()]
    ev, et = max(wV, 1e-6), max(wT, 1e-6)                    # avoid division by zero inside WBF when a weight is 0
    sl = [(pV[1] * ev).tolist(), (pT[1] * et).tolist()]; ll = [pV[2].astype(float).tolist(), pT[2].astype(float).tolist()]
    if len(sl[0]) + len(sl[1]) == 0: out = (np.zeros((0, 4), np.float32), np.zeros(0, np.float32), np.zeros(0, int))
    else:
        b, s, l = weighted_boxes_fusion(bl, sl, ll, weights=None, iou_thr=WBF_IOU, skip_box_thr=WBF_SKIP, conf_type=WBF_CONF_TYPE)
        o = np.argsort(-s, kind="stable")[:MAX_DET_EVAL]; out = (b[o].astype(np.float32), s[o].astype(np.float32), l[o].astype(int))
    _fuse_cache[key] = out; return out

def get_pred(spec, i):
    k = spec[0]
    if k == "V": return PRED["V"][i]
    if k == "T": return PRED["T"][i]
    if k == "static": return fuse(i, spec[1], round(1 - spec[1], 5))
    if k == "staticfold": w = dict(spec[1])[ALL[i]["fold"]]; return fuse(i, w, round(1 - w, 5))      # weight chosen per held-out fold
    if k == "staticmap": w = dict(spec[1])[ALL[i]["cond"]]; return fuse(i, w, round(1 - w, 5))        # weight per time window (oracle analyses)
    if k == "alf": wV, wT = alf_weights(ALL[i]["muV"], ALL[i]["CT"], spec[2], spec[1]); return fuse(i, wV, wT)

IOUV = np.linspace(0.5, 0.95, 10)
_trapz = getattr(np, "trapezoid", None) or np.trapz
def iou_matrix(a, b):
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

def match_image(P, G, thrs=IOUV):
    """Greedy matching of predictions (descending confidence) to ground truth at each IoU threshold."""
    pb, ps, pc = P; gb, gc = G
    o = np.argsort(-ps, kind="stable"); pb, ps, pc = pb[o], ps[o], pc[o]
    tp = np.zeros((len(ps), len(thrs)), bool)
    if len(ps) and len(gc):
        iou = iou_matrix(pb, gb) * (pc[:, None] == gc[None, :])
        for k, t in enumerate(thrs):
            used = np.zeros(len(gc), bool)
            for n in range(len(ps)):
                cand = np.where(used, -1.0, iou[n]); j = int(cand.argmax())
                if cand[j] >= t: used[j] = True; tp[n, k] = True
    return dict(conf=ps, cls=pc, tp=tp, ngt=np.bincount(gc.astype(int), minlength=NC)[:NC])

def compute_ap(rec, prec):                   # 101-point interpolated AP
    mrec = np.concatenate(([0.0], rec, [1.0])); mpre = np.concatenate(([1.0], prec, [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101); return float(_trapz(np.interp(x, mrec, mpre), x))

def evaluate(ms, thrs=IOUV):
    conf = np.concatenate([m["conf"] for m in ms]); cls = np.concatenate([m["cls"] for m in ms])
    tp = np.concatenate([m["tp"] for m in ms], 0) if len(conf) else np.zeros((0, len(thrs)), bool)
    ngt = np.sum([m["ngt"] for m in ms], 0)
    classes = [c for c in range(NC) if ngt[c] > 0]; aps = np.zeros((len(classes), len(thrs)))
    for ci, c in enumerate(classes):
        s = np.where(cls == c)[0]
        if len(s) == 0: continue
        o = s[np.argsort(-conf[s], kind="stable")]; t = tp[o]
        tpc, fpc = np.cumsum(t, 0), np.cumsum(~t, 0)
        rec, prec = tpc / ngt[c], tpc / np.maximum(tpc + fpc, 1)
        for k in range(len(thrs)): aps[ci, k] = compute_ap(rec[:, k], prec[:, k])
    res = dict(mAP50=aps[:, 0].mean(), mAP5095=aps.mean())
    o = np.argsort(-conf, kind="stable"); t0 = tp[o][:, 0] if len(o) else np.zeros(0, bool)
    if len(t0):                                # precision / recall at the F1-optimal confidence threshold (IoU 0.5)
        tpc, fpc = np.cumsum(t0), np.cumsum(~t0)
        R = tpc / max(ngt.sum(), 1); Pp = tpc / np.maximum(tpc + fpc, 1); F = 2 * Pp * R / np.maximum(Pp + R, 1e-9)
        i = int(F.argmax()); res.update(P=Pp[i], R=R[i], F1=F[i], conf_opt=float(conf[o][i]))
    else: res.update(P=0., R=0., F1=0., conf_opt=0.5)
    return res

def skey(spec): return ("alf", spec[1], tuple(sorted(spec[2].items()))) if spec[0] == "alf" else tuple(spec)
_mc = {}
def matches(spec, idxs, thrs=IOUV):
    out = []
    for i in idxs:
        k = (skey(spec), i, len(thrs))
        if k not in _mc: _mc[k] = match_image(get_pred(spec, i), (ALL[i]["gt_boxes"], ALL[i]["gt_cls"]), thrs)
        out.append(_mc[k])
    return out
def map50(spec, idxs): return evaluate(matches(spec, idxs, np.array([0.5])), np.array([0.5]))["mAP50"]
def pct(x): return round(float(x) * 100, DECIMALS)

## 7. Static-weight baseline: selected on the other folds

In [ ]:
# "Best static" baseline: for each held-out fold k, the visual weight w_V (grid WGRID) that maximises mAP@0.5 on the out-of-fold
# predictions of the OTHER folds. The held-out fold never selects its own weight.
WMAP, rows = {}, []
for k_ in sorted({r["fold"] for r in ALL}):
    others = [i for i in TEST_IDX if ALL[i]["fold"] != k_]
    sc = {float(w): map50(("static", float(w)), others) for w in WGRID}
    wbest = max(sc, key=lambda w: (sc[w], -abs(w - 0.5))); WMAP[k_] = wbest
    rows.append(dict(held_out_fold=k_, selected_wV=wbest, mAP50_on_other_folds=round(sc[wbest] * 100, 1)))
T_STATIC = pd.DataFrame(rows); display(T_STATIC)
STATIC_SPEC = ("staticfold", tuple(sorted(WMAP.items())))
STATIC_DESC = "w_V per fold: " + ", ".join(f"fold {k}={w:.2f}" for k, w in WMAP.items())
print(STATIC_DESC)
print("Note: the detectors that produced the other folds' predictions were trained with data of the held-out fold (mild stacking-type leakage).")

## 8. Overall performance with session-level bootstrap intervals

In [ ]:
N_V, N_T, N_F5, N_FB, N_ALF = "Visual-only (YOLO11n)", "Thermal-only (YOLO11n)", "Fixed WBF (0.5/0.5)", "Fixed WBF (best static)", "ALF"
N_X = "ALF, max-normalised weights [exploratory, post hoc]"
SPEC = OrderedDict([(N_V, ("V",)), (N_T, ("T",)), (N_F5, ("static", 0.5)), (N_FB, STATIC_SPEC), (N_ALF, ("alf", "full", ALFP))])
SPEC_ABL = OrderedDict([("Fixed WBF (0.5/0.5)", ("static", 0.5)), ("+ VQE only", ("alf", "vqe", ALFP)),
                        ("+ TQE only", ("alf", "tqe", ALFP)), ("+ Both (full ALF)", ("alf", "full", ALFP))])
SPEC_EXPL = OrderedDict([(N_X, ("alf", "norm", ALFP))])
M = {n: matches(sp, TEST_IDX) for n, sp in {**SPEC, **SPEC_ABL, **SPEC_EXPL}.items()}       # per-image matches (reused everywhere)
E = {n: evaluate(m) for n, m in M.items()}
CONF_OPT = {n: E[n]["conf_opt"] for n in E}
def sub(n, pos): return [M[n][k] for k in pos]

T_OVERALL = pd.DataFrame({n: {"mAP@0.5 (%)": pct(E[n]["mAP50"]), "mAP@0.5:0.95 (%)": pct(E[n]["mAP5095"]),
                              "Precision (%)": pct(E[n]["P"]), "Recall (%)": pct(E[n]["R"])} for n in SPEC}).T
T_OVERALL.index.name = "Method"; display(T_OVERALL)
print("Fixed (best static):", STATIC_DESC, "| precision/recall at the F1-optimal confidence threshold of each method (IoU 0.5).")

In [ ]:
def boot_prep(ms):
    conf = np.concatenate([m["conf"] for m in ms]); cls = np.concatenate([m["cls"] for m in ms])
    tp = np.concatenate([m["tp"][:, 0] for m in ms]); img = np.concatenate([np.full(len(m["conf"]), k, int) for k, m in enumerate(ms)])
    ngt = np.stack([m["ngt"] for m in ms]); prep = []
    for c in range(NC):
        s = np.where(cls == c)[0]; o = s[np.argsort(-conf[s], kind="stable")]; prep.append((img[o], tp[o], ngt[:, c]))
    return prep
def boot_ap50(prep, counts):                 # AP@0.5 for a resample given as per-image multiplicities
    aps = []
    for img, tp, ng in prep:
        n = (counts * ng).sum()
        if n == 0: continue
        w = counts[img]; m = w > 0; w, t = w[m], tp[m]
        if len(w) == 0: aps.append(0.0); continue
        tpc, fpc = np.cumsum(w * t), np.cumsum(w * (~t)); aps.append(compute_ap(tpc / n, tpc / np.maximum(tpc + fpc, 1e-9)))
    return float(np.mean(aps)) if aps else 0.0

PREP = {n: boot_prep(m) for n, m in M.items()}
assert abs(boot_ap50(PREP[N_ALF], np.ones(len(TEST_IDX), int)) - E[N_ALF]["mAP50"]) < 1e-6, "bootstrap inconsistent with evaluate()"
rng = np.random.default_rng(SEED)
def make_counts(groups):
    """One bootstrap resample as per-image multiplicities. groups: list of position arrays (one stratum each). Sessions are resampled within a stratum."""
    c = np.zeros(len(TEST_IDX), int)
    for g in groups:
        g = np.asarray(g)
        if BOOT_UNIT == "session":
            ses = sorted({SESS[k] for k in g}); members = {s_: [k for k in g if SESS[k] == s_] for s_ in ses}
            for p in rng.choice(len(ses), len(ses), replace=True): c[members[ses[p]]] += 1
        else: c += np.bincount(rng.choice(g, len(g), replace=True), minlength=len(TEST_IDX))
    return c
STRAT = [np.array(POS[c]) for c in COND_MAP]                 # stratified by time window
BOOT_ALL = [make_counts(STRAT) for _ in range(N_BOOT)]
def boot_dist(names, counts_list): return {n: np.array([boot_ap50(PREP[n], c) for c in counts_list]) for n in names}
BD = boot_dist(list(M), BOOT_ALL)
ci = lambda a: (np.percentile(a, 2.5) * 100, np.percentile(a, 97.5) * 100)

T_OVERALL_CI = pd.DataFrame({n: {"mAP@0.5 (%)": f"{pct(E[n]['mAP50']):.{DECIMALS}f}", "95% CI": "[{:.{d}f}, {:.{d}f}]".format(*ci(BD[n]), d=DECIMALS)} for n in SPEC}).T
T_OVERALL_CI.index.name = "Method"; display(T_OVERALL_CI)

def paired(a, b):
    d = (BD[a] - BD[b]) * 100
    return {"Delta mAP@0.5 (pp)": round((E[a]["mAP50"] - E[b]["mAP50"]) * 100, 2), "95% CI (pp)": "[{:.2f}, {:.2f}]".format(*np.percentile(d, [2.5, 97.5])),
            "p (two-sided bootstrap)": round(float(min(1, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1 / N_BOOT)), 4)}
PAIRS = [(N_ALF, N_V), (N_ALF, N_T), (N_ALF, N_F5), (N_ALF, N_FB), (N_F5, N_T), (N_F5, N_V), (N_X, N_F5), (N_X, N_ALF), (N_X, N_T)]
T_PAIRED = pd.DataFrame({f"{a} vs {b}": paired(a, b) for a, b in PAIRS}).T; T_PAIRED.index.name = "Comparison (A vs B)"; display(T_PAIRED)
print(f"Bootstrap unit: {BOOT_UNIT} (stratified by time window, {N_BOOT} resamples). With few sessions the intervals are wide; paired differences use the same resamples.")

## 9. Results by time window

In [ ]:
cols_w = OrderedDict([("Visual-only", N_V), ("Thermal-only", N_T), ("Fixed WBF (0.5/0.5)", N_F5), ("Fixed WBF (best static)", N_FB), ("ALF", N_ALF)])
T_WINDOW = pd.DataFrame({c: {g: pct(evaluate(sub(n, sum([POS[x] for x in conds], [])))["mAP50"]) for g, conds in GROUPS.items()} for c, n in cols_w.items()})
T_WINDOW.index.name = "Time window (dawn and dusk pooled)"; display(T_WINDOW)

rows = []
for c, cn in COND_MAP.items():
    for lab in SPEC:
        e = evaluate(sub(lab, POS[c])); rows.append({"Window": cn, "Method": lab, "mAP@0.5 (%)": pct(e["mAP50"]), "mAP@0.5:0.95 (%)": pct(e["mAP5095"]), "Precision (%)": pct(e["P"]), "Recall (%)": pct(e["R"])})
T_ALLMETRICS = pd.DataFrame(rows)
T_FOUR = T_ALLMETRICS.pivot(index="Method", columns="Window", values="mAP@0.5 (%)").loc[list(SPEC), list(COND_MAP.values())]
display(T_FOUR)

rows = {}
for g, conds in GROUPS.items():
    pos = np.array(sum([POS[x] for x in conds], [])); cl = [make_counts([pos]) for _ in range(N_BOOT)]
    bd = boot_dist(list(cols_w.values()), cl)
    rows[g] = {c: "{:.{d}f} [{:.{d}f}, {:.{d}f}]".format(T_WINDOW.loc[g, c], *ci(bd[n]), d=DECIMALS) for c, n in cols_w.items()}
T_WINDOW_CI = pd.DataFrame(rows).T; T_WINDOW_CI.index.name = "Time window"; display(T_WINDOW_CI)

## 10. Ablation: effect of each quality estimator

In [ ]:
T_ABL = pd.DataFrame({"mAP@0.5 (%)": {n: pct(E[n]["mAP50"]) for n in SPEC_ABL}}); T_ABL.index.name = "Configuration"
T_ABL["Delta vs fixed (pp)"] = [round((E[n]["mAP50"] - E["Fixed WBF (0.5/0.5)"]["mAP50"]) * 100, 2) for n in SPEC_ABL]
T_ABL["95% CI of mAP@0.5"] = ["[{:.{d}f}, {:.{d}f}]".format(*ci(BD[n]), d=DECIMALS) for n in SPEC_ABL]
display(T_ABL)
rows = {g: {n: pct(evaluate(sub(n, sum([POS[x] for x in conds], [])))["mAP50"]) for n in SPEC_ABL} for g, conds in GROUPS.items()}
T_ABL_WINDOW = pd.DataFrame(rows); T_ABL_WINDOW.index.name = "Configuration"; display(T_ABL_WINDOW)
print("VQE only: thermal quality fixed at Q_T,high. TQE only: visual quality fixed at Q_V,high.")

## 11. Quality-estimator statistics and adaptive weights by window

In [ ]:
rows = []
for c, cn in list(COND_MAP.items()) + [("ALL", "All test")]:
    idx = TEST_IDX if c == "ALL" else [i for i in TEST_IDX if ALL[i]["cond"] == c]
    W = np.array([alf_weights(ALL[i]["muV"], ALL[i]["CT"], ALFP) for i in idx])
    mu = np.array([ALL[i]["muV"] for i in idx]); ct = np.array([ALL[i]["CT"] for i in idx])
    rows.append({"Window": cn, "N": len(idx), "mean muV": mu.mean().round(3), "mean C_T": ct.mean().round(3),
                 "VQE penalises RGB (%)": round(((mu < ALFP["tau_dark"]) | (mu > ALFP["tau_bright"])).mean() * 100, 1),
                 "TQE flags crossover (%)": round((ct < ALFP["tau_cross"]).mean() * 100, 1),
                 "mean w_V": W[:, 0].mean().round(3), "mean w_T": W[:, 1].mean().round(3)})
T_EST = pd.DataFrame(rows).set_index("Window"); display(T_EST)

## 12. Fixed-weight sweeps within each window, and oracle headroom (uses test labels; descriptive)

In [ ]:
rows = []
for c, cn in COND_MAP.items():
    idx = [i for i in TEST_IDX if ALL[i]["cond"] == c]
    for w in WGRID: rows.append(dict(window=cn, w_V=float(w), mAP50=map50(("static", float(w)), idx) * 100))
SWEEP = pd.DataFrame(rows)
plt.figure(figsize=(6.4, 4))
for cn, g_ in SWEEP.groupby("window", sort=False): plt.plot(g_.w_V, g_.mAP50, "o-", ms=3, label=cn)
plt.xlabel("Visual weight $w_V$  ($w_T = 1 - w_V$)"); plt.ylabel("mAP@0.5 (%)"); plt.grid(alpha=.3); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_weight_sweep_by_window.png", dpi=300); plt.show()

best_w = SWEEP.loc[SWEEP.groupby("window", sort=False).mAP50.idxmax()].set_index("window")[["w_V", "mAP50"]].round(1)
best_w.columns = ["oracle w_V", "oracle mAP@0.5"]
best_w["ALF mAP@0.5"] = pd.Series({COND_MAP[c]: evaluate(sub(N_ALF, POS[c]))["mAP50"] * 100 for c in COND_MAP}).round(1)
best_w["Fixed 0.5 mAP@0.5"] = pd.Series({COND_MAP[c]: evaluate(sub(N_F5, POS[c]))["mAP50"] * 100 for c in COND_MAP}).round(1)
best_w["ALF mean w_V"] = [T_EST.loc[k, "mean w_V"] for k in best_w.index]
T_ORACLE = best_w; display(T_ORACLE)

_inv = {v: k for k, v in COND_MAP.items()}
oracle_map = tuple((_inv[w], float(T_ORACLE.loc[w, "oracle w_V"])) for w in T_ORACLE.index)
POOLED_ORACLE_WEIGHTS = evaluate(matches(("staticmap", oracle_map), TEST_IDX))["mAP50"] * 100
choice = {}
for c in COND_MAP:
    mv_, mt_ = evaluate(sub(N_V, POS[c]))["mAP50"], evaluate(sub(N_T, POS[c]))["mAP50"]; choice[c] = "V" if mv_ >= mt_ else "T"
POOLED_ORACLE_SELECTION = evaluate([M[N_V if choice[ALL[i]["cond"]] == "V" else N_T][k] for k, i in enumerate(TEST_IDX)])["mAP50"] * 100
print(f"Pooled mAP@0.5 with oracle per-window weights: {POOLED_ORACLE_WEIGHTS:.1f} | with the best single modality per window: {POOLED_ORACLE_SELECTION:.1f} "
      f"| ALF: {E[N_ALF]['mAP50']*100:.1f} | fixed 0.5/0.5: {E[N_F5]['mAP50']*100:.1f}")

## 13. Per-window versus pooled evaluation (score-scale diagnostic)

In [ ]:
# ALF multiplies detector confidences by condition-dependent weights, which changes the confidence scale between conditions.
# A pooled ranking (single global threshold) can therefore differ from per-window rankings. The max-normalised variant keeps the
# dominant modality's confidence scale and is an exploratory, post hoc diagnostic only.
SPEC_DIAG = OrderedDict([("Fixed WBF (0.5/0.5)", ("static", 0.5)), ("ALF", ("alf", "full", ALFP)), ("ALF, max-normalised (post hoc)", ("alf", "norm", ALFP))])
MD_ = {n: matches(sp, TEST_IDX) for n, sp in SPEC_DIAG.items()}
rows = {}
for n in SPEC_DIAG:
    r_ = {COND_MAP[c]: pct(evaluate([MD_[n][k] for k in POS[c]])["mAP50"]) for c in COND_MAP}
    r_["Mean of four windows"] = round(float(np.mean(list(r_.values()))), DECIMALS); r_["Pooled (all test pairs)"] = pct(evaluate(MD_[n])["mAP50"]); rows[n] = r_
orc = {COND_MAP[c]: float(T_ORACLE.loc[COND_MAP[c], "oracle mAP@0.5"]) for c in COND_MAP}
rows["Oracle w_V per window (test labels)"] = {**orc, "Mean of four windows": round(float(np.mean(list(orc.values()))), DECIMALS), "Pooled (all test pairs)": round(POOLED_ORACLE_WEIGHTS, DECIMALS)}
T_PER_WINDOW = pd.DataFrame(rows).T; T_PER_WINDOW.index.name = "Method"; display(T_PER_WINDOW)

## 14. Source-only versus cross-validated detectors

In [ ]:
T_SOURCE_VS_CV = None
if HAVE_SOURCE and PRED_SRC is not None:
    _keep = (PRED, dict(_mc), dict(_fuse_cache)); PRED = PRED_SRC; _mc.clear(); _fuse_cache.clear()
    SRC = {n: evaluate(matches(sp_, TEST_IDX)) for n, sp_ in [(N_V, ("V",)), (N_T, ("T",)), (N_F5, ("static", 0.5)), (N_ALF, ("alf", "full", ALFP))]}
    PRED = _keep[0]; _mc.clear(); _mc.update(_keep[1]); _fuse_cache.clear(); _fuse_cache.update(_keep[2])
    T_SOURCE_VS_CV = pd.DataFrame({n: {"Source-only mAP@0.5 (%)": pct(SRC[n]["mAP50"]), "Cross-validated mAP@0.5 (%)": pct(E[n]["mAP50"]),
                                       "Gain (pp)": round((E[n]["mAP50"] - SRC[n]["mAP50"]) * 100, 1)} for n in SRC}).T
    T_SOURCE_VS_CV.index.name = "Method"; display(T_SOURCE_VS_CV)
    print("Source-only: detectors trained on the original training pairs only. Cross-validated: detectors that also saw the test sessions of the other folds (same scene).")
else:
    print("Skipped (source-only weights not available).")

## 15. Sensitivity of ALF to its parameters (test set; robustness check, not for model selection)

In [ ]:
SENS = {}
if RUN_SENSITIVITY:
    t0 = time.time(); vals = [0.1, 0.3, 0.5, 0.7]; H = np.zeros((len(vals), len(vals)))
    for a, qv in enumerate(vals):
        for b, qt in enumerate(vals): H[a, b] = map50(("alf", "full", {**ALFP, "QV_low": qv, "QT_low": qt}), TEST_IDX) * 100
    SENS["heat"] = pd.DataFrame(H, index=[f"Q_V,low={v}" for v in vals], columns=[f"Q_T,low={v}" for v in vals])
    fig, ax = plt.subplots(figsize=(4.6, 3.8)); im = ax.imshow(H, cmap="viridis")
    ax.set_xticks(range(len(vals))); ax.set_xticklabels(vals); ax.set_yticks(range(len(vals))); ax.set_yticklabels(vals)
    ax.set_xlabel("$Q_{T,low}$"); ax.set_ylabel("$Q_{V,low}$"); ax.set_title("mAP@0.5 (%) of ALF", fontsize=10)
    for (y, x), v in np.ndenumerate(H): ax.text(x, y, f"{v:.1f}", ha="center", va="center", color="w", fontsize=8)
    plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig_sensitivity_quality_levels.png", dpi=300); plt.show()

    sweeps = {"tau_dark": [0.05, 0.1, 0.15, 0.2, 0.3, 0.4], "tau_bright": [0.6, 0.7, 0.85, 0.9, 0.95], "tau_cross": [0.05, 0.1, 0.25, 0.4, 0.55, 0.7, 0.85, 0.95]}
    rows = []
    for k, vs in sweeps.items():
        for v in vs:
            frac = (QDF.CT < v).mean() if k == "tau_cross" else ((QDF.muV < v).mean() if k == "tau_dark" else (QDF.muV > v).mean())
            rows.append({"parameter": k, "value": v, "frames triggered (%)": round(frac * 100, 1), "mAP@0.5 (%)": round(map50(("alf", "full", {**ALFP, k: v}), TEST_IDX) * 100, 2)})
    SENS["sweep"] = pd.DataFrame(rows); display(SENS["sweep"])
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
    for a_, (k, g_) in zip(ax, SENS["sweep"].groupby("parameter", sort=False)):
        a_.plot(g_.value, g_["mAP@0.5 (%)"], "s-"); a_.axvline(ALFP[k], color="r", ls="--", lw=1); a_.set_xlabel(k); a_.set_ylabel("mAP@0.5 (%)"); a_.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig_sensitivity_thresholds.png", dpi=300); plt.show()
    print(f"Done in {time.time()-t0:.0f}s. Red dashed line = value used in the main analysis.")

## 16. Processing time per frame

In [ ]:
T_LATENCY = None
if RUN_LATENCY:
    cand_v = [p for p in (OOF_FILE.parent / "fold0" / "visual_last.pt", W_VIS) if p is not None and Path(p).exists()]
    cand_t = [p for p in (OOF_FILE.parent / "fold0" / "thermal_last.pt", W_THR) if p is not None and Path(p).exists()]
    if cand_v and cand_t:
        lat_v, lat_t = YOLO(str(cand_v[0])), YOLO(str(cand_t[0]))
        sel = TEST_IDX[:: max(1, len(TEST_IDX) // 30)][:30]
        for i_ in sel[:3]: lat_v.predict(str(ALL[i_]["vis"]), imgsz=IMGSZ, device=DEVICE, verbose=False)      # warm-up
        def timeit(fn):
            t = []
            for i in sel: s = time.perf_counter(); fn(i); t.append((time.perf_counter() - s) * 1000)
            return np.mean(t), np.std(t)
        det_v = timeit(lambda i: lat_v.predict(str(ALL[i]["vis"]), imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False))
        det_t = timeit(lambda i: lat_t.predict(str(ALL[i]["thr"]), imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False))
        imgs = {i: (cv2.imread(str(ALL[i]["vis"])), thermal_gray01(ALL[i]["thr"])) for i in sel}
        qe = timeit(lambda i: (mu_v(imgs[i][0]), michelson(imgs[i][1])))
        def _wbf(i):
            wV, wT = alf_weights(ALL[i]["muV"], ALL[i]["CT"], ALFP); pV, pT = PRED["V"][i], PRED["T"][i]
            k = lambda p, w: ([np.clip(p[0][p[1] > 0.25], 0, 1).tolist()], [(p[1][p[1] > 0.25] * w).tolist()], [p[2][p[1] > 0.25].astype(float).tolist()])
            a, b = k(pV, wV), k(pT, wT)
            if len(a[1][0]) + len(b[1][0]): weighted_boxes_fusion(a[0] + b[0], a[1] + b[1], a[2] + b[2], iou_thr=WBF_IOU, conf_type=WBF_CONF_TYPE)
        wb = timeit(_wbf)
        T_LATENCY = pd.DataFrame({"ms per frame (mean ± SD)": {"YOLO11n visual detector": f"{det_v[0]:.1f} ± {det_v[1]:.1f}", "YOLO11n thermal detector": f"{det_t[0]:.1f} ± {det_t[1]:.1f}",
                                  "Quality estimation (VQE + TQE)": f"{qe[0]:.2f} ± {qe[1]:.2f}", "Adaptive WBF fusion (confidence > 0.25)": f"{wb[0]:.2f} ± {wb[1]:.2f}"}})
        T_LATENCY.index.name = "Component"; display(T_LATENCY)
        print(f"Device: {torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'}, {len(sel)} frames; detector time includes file reading, pre-processing and NMS.")
        print(f"Quality-estimation overhead = {qe[0] / (det_v[0] + det_t[0]) * 100:.2f}% of the two detectors' time.")
    else: print("No detector weights found for the timing test.")

## 17. Qualitative examples (one test frame per window)

In [ ]:
def frame_score(name, k):                    # (true positives - false positives) in a frame at the method's F1-optimal threshold
    m = M[name][k]; s = m["conf"] >= CONF_OPT[name]; t = int(m["tp"][s, 0].sum()); return t - (int(s.sum()) - t)
rs_ = np.random.default_rng(SEED); QUAL = {}
for c in COND_MAP:
    cand = [k for k in POS[c] if len(ALL[TEST_IDX[k]]["gt_cls"]) > 0]
    QUAL[c] = max(cand, key=lambda k: frame_score(N_ALF, k) - frame_score(N_F5, k)) if QUAL_MODE == "max_gain" else int(rs_.choice(cand))
def draw(ax, img, P, thr, gt, title):
    ax.imshow(img); H, W = img.shape[:2]
    for b in gt: ax.add_patch(plt.Rectangle((b[0]*W, b[1]*H), (b[2]-b[0])*W, (b[3]-b[1])*H, fill=False, ec="lime", lw=1.6, ls="--"))
    keep = P[1] >= thr
    for b, s in zip(P[0][keep], P[1][keep]):
        ax.add_patch(plt.Rectangle((b[0]*W, b[1]*H), (b[2]-b[0])*W, (b[3]-b[1])*H, fill=False, ec="red", lw=1.6))
        ax.text(b[0]*W, b[1]*H, f"{s:.2f}", color="w", fontsize=6, bbox=dict(fc="red", ec="none", pad=0.5))
    ax.set_title(title, fontsize=7); ax.axis("off")
fig, axes = plt.subplots(len(COND_MAP), 4, figsize=(12, 7.2)); rowsq = []
for r_, c in enumerate(COND_MAP):
    i = TEST_IDX[QUAL[c]]; R = ALL[i]; wV, wT = alf_weights(R["muV"], R["CT"], ALFP)
    vis = cv2.cvtColor(cv2.imread(str(R["vis"])), cv2.COLOR_BGR2RGB); th = cv2.cvtColor(cv2.imread(str(R["thr"])), cv2.COLOR_BGR2RGB)
    draw(axes[r_, 0], vis, PRED["V"][i], CONF_OPT[N_V], R["gt_boxes"], f"{COND_MAP[c]} - visual-only ($\\mu_V$={R['muV']:.2f})")
    draw(axes[r_, 1], th, PRED["T"][i], CONF_OPT[N_T], R["gt_boxes"], f"thermal-only ($C_T$={R['CT']:.2f})")
    draw(axes[r_, 2], vis, get_pred(SPEC[N_F5], i), CONF_OPT[N_F5], R["gt_boxes"], "fixed WBF (0.5/0.5)")
    draw(axes[r_, 3], vis, get_pred(SPEC[N_ALF], i), CONF_OPT[N_ALF], R["gt_boxes"], f"ALF ($w_V$={wV:.2f}, $w_T$={wT:.2f})")
    rowsq.append(dict(window=COND_MAP[c], frame=R["vis"].name, muV=round(R["muV"], 3), CT=round(R["CT"], 3), w_V=round(wV, 3), w_T=round(wT, 3)))
plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig_qualitative.png", dpi=300, bbox_inches="tight"); plt.show()
T_QUAL = pd.DataFrame(rowsq).set_index("window"); display(T_QUAL)
print(f"Frame selection: {QUAL_MODE}. Green dashed = ground truth; red = detections at each method's F1-optimal threshold.")

## 18. Export results

In [ ]:
import importlib.metadata as im
TABLES = OrderedDict([("overall_performance", T_OVERALL), ("overall_performance_ci", T_OVERALL_CI), ("paired_comparisons", T_PAIRED),
                      ("by_time_window", T_WINDOW), ("by_time_window_ci", T_WINDOW_CI), ("by_window_four", T_FOUR), ("all_metrics_by_window", T_ALLMETRICS.set_index(["Window", "Method"])),
                      ("ablation", T_ABL), ("ablation_by_window", T_ABL_WINDOW), ("estimator_statistics", T_EST), ("dataset_statistics", T_DATA.set_index("Subset")),
                      ("static_weight_selection", T_STATIC), ("weight_sweep_by_window", SWEEP.set_index("window")), ("oracle_headroom", T_ORACLE),
                      ("per_window_vs_pooled", T_PER_WINDOW), ("qualitative_frames", T_QUAL)])
if T_SOURCE_VS_CV is not None: TABLES["source_only_vs_cv"] = T_SOURCE_VS_CV
if T_LATENCY is not None: TABLES["latency"] = T_LATENCY
if SENS.get("sweep") is not None: TABLES["sensitivity_thresholds"] = SENS["sweep"].set_index("parameter"); TABLES["sensitivity_quality_levels"] = SENS["heat"]
with pd.ExcelWriter(RESULTS_DIR / "results.xlsx") as xw:
    for k, df in TABLES.items(): df.to_excel(xw, sheet_name=k[:31]); df.to_csv(RESULTS_DIR / f"{k}.csv")
def ver(p):
    try: return im.version(p)
    except Exception: return "?"
CFG = dict(ALF_parameters=ALFP, tqe_percentile=TQE_PERCENTILE, imgsz=IMGSZ, conf_pred=CONF_PRED, nms_iou=NMS_IOU, max_det=MAX_DET, wbf_iou=WBF_IOU, wbf_conf_type=WBF_CONF_TYPE,
           static_weight_selection=STATIC_DESC, n_bootstrap=N_BOOT, bootstrap_unit=BOOT_UNIT, seed=SEED, n_test_pairs=len(ALL), oof_file=str(OOF_FILE),
           versions={p: ver(p) for p in ("ultralytics", "torch", "numpy", "ensemble-boxes", "opencv-python")}, python=platform.python_version())
(RESULTS_DIR / "config_used.json").write_text(json.dumps(CFG, indent=2, default=str))
QDF.to_csv(RESULTS_DIR / "quality_scores_per_frame.csv", index=False)
pi = []
for k, i in enumerate(TEST_IDX):
    row = dict(frame=ALL[i]["vis"].name, window=COND_MAP[ALL[i]["cond"]], session=ALL[i]["session"], fold=ALL[i]["fold"], n_boxes=len(ALL[i]["gt_cls"]), muV=ALL[i]["muV"], CT=ALL[i]["CT"])
    row["w_V"], row["w_T"] = alf_weights(ALL[i]["muV"], ALL[i]["CT"], ALFP)
    for n in SPEC: row[f"TP_minus_FP[{n}]"] = frame_score(n, k)
    pi.append(row)
pd.DataFrame(pi).to_csv(RESULTS_DIR / "per_frame_results.csv", index=False)
print("Saved to:", RESULTS_DIR); print(sorted(p.name for p in RESULTS_DIR.iterdir()))